### Load the libraries

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [50]:
### Load the Dataset

In [51]:
df = pd.read_csv(r'D:\CUSTOMER-CHURN\Dataset\churn_dataset_cleaned.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [52]:
### Dataset overview

In [53]:
df.shape

(7043, 20)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


### Problem Statement
* Build a system which can take features of a Telecom Customer like tenure, MonthlyCharges, Contract type, InternetService, etc., and predict whether they will Churn.
* Target variable - Churn (Yes/No mapped to 1/0)
* Evaluation Metric - Accuracy Score


### Data Preprocessing

In [55]:
## 1. Strictly drop customerID and TotalCharges
cols_to_drop = [col for col in ['customerID', 'TotalCharges'] if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

## 2. Encode Target Variable
df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})

## 3. Separate the X and Y
Y = df['Churn'] 
X = df.drop(columns=['Churn'])

## 4. Automated Categorical Encoding using pd.get_dummies
X = pd.get_dummies(X, drop_first=True, dtype=int)

## 5. Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42, stratify=Y)

## Save column names for deployment later
colos = X_train.columns.tolist()

X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
6661,0,72,53.65,0,1,1,0,0,0,0,...,0,1,1,1,0,1,0,1,0,0
4811,0,4,46.00,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
2193,0,56,21.20,1,0,1,1,0,0,1,...,0,0,0,0,1,0,1,0,0,1
1904,0,56,94.45,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
6667,0,9,79.55,0,0,0,1,0,1,0,...,0,0,0,1,0,0,1,0,1,0


In [56]:
## 6. Standardization (ONLY on continuous numerical features)
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()

# Make a copy to avoid overwriting the original split variables
X_train_transformed = X_train.copy()
X_test_transformed = X_test.copy()

# Define purely continuous columns
num_cols = ['tenure', 'MonthlyCharges']

# Fit the scaler ONLY on the numeric columns of the training data
X_train_transformed[num_cols] = scale.fit_transform(X_train[num_cols])

# Transform the numeric columns of the test data
X_test_transformed[num_cols] = scale.transform(X_test[num_cols])

print("Shape of X_train_transformed:", X_train_transformed.shape)
display(X_train_transformed.head())

Shape of X_train_transformed: (5282, 22)


,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
6661,0,1.609608,-0.371461,0,1,1,0,0,0,0,...,0,1,1,1,0,1,0,1,0,0
4811,0,-1.151780,-0.625032,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
2193,0,0.959870,-1.447065,1,0,1,1,0,0,1,...,0,0,0,0,1,0,1,0,0,1
1904,0,0.959870,0.980917,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
6667,0,-0.948737,0.487034,0,0,0,1,0,1,0,...,0,0,0,1,0,0,1,0,1,0


## Model Building
1. Logistic Regression

In [57]:
from sklearn.linear_model import LogisticRegression

LogReg = LogisticRegression(max_iter=1000, random_state=42)
LogReg.fit(X_train_transformed, Y_train)

## making prediction
y_pred_LR = LogReg.predict(X_test_transformed)

In [58]:
from sklearn import metrics
acc_LR = metrics.accuracy_score(Y_test, y_pred_LR)
print('Accuracy Score: ', acc_LR)

Accuracy Score:  0.7995457126632595


### KNN

In [59]:
from sklearn.neighbors import KNeighborsClassifier

KNNclf = KNeighborsClassifier()
KNNclf.fit(X_train_transformed, Y_train)

## Making Predictions
y_pred_KNN = KNNclf.predict(X_test_transformed)

In [60]:
acc_KNN = metrics.accuracy_score(Y_test, y_pred_KNN)
print('Accuracy Score: ', acc_KNN)

Accuracy Score:  0.7643384440658717


### Decision Tree

In [61]:
from sklearn.tree import DecisionTreeClassifier

DTclf = DecisionTreeClassifier(random_state=42)
DTclf.fit(X_train_transformed, Y_train)

## making predictions
y_pred_DT = DTclf.predict(X_test_transformed)

In [62]:
### Evaluations
acc_DT = metrics.accuracy_score(Y_test, y_pred_DT)
print('Accuracy Score: ', acc_DT)

Accuracy Score:  0.7240204429301533


### Random Forest

In [63]:
from sklearn.ensemble import RandomForestClassifier

RFclf = RandomForestClassifier(random_state=42)
RFclf.fit(X_train_transformed, Y_train)

### Making predictions
y_pred_RF = RFclf.predict(X_test_transformed)

In [64]:
### Evaluations
acc_RF = metrics.accuracy_score(Y_test, y_pred_RF)
print('Accuracy Score: ', acc_RF)

Accuracy Score:  0.7853492333901193


### Support Vector Machines

In [65]:
from sklearn.svm import SVC

SVMclf = SVC(random_state=42)
SVMclf.fit(X_train_transformed, Y_train)

### making the predictions
y_pred_SVM = SVMclf.predict(X_test_transformed)

In [66]:
### Evaluations
acc_SVM = metrics.accuracy_score(Y_test, y_pred_SVM)
print('Accuracy Score: ', acc_SVM)

Accuracy Score:  0.7978421351504826


### Evaluation and Comparison

In [67]:
evaluation = pd.DataFrame({
    'Algorithms': ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'Support Vector Machines'], 
    'Accuracy': [acc_LR, acc_KNN, acc_DT, acc_RF, acc_SVM]
})
evaluation.sort_values(by='Accuracy', ascending=False, inplace=True)
evaluation.reset_index(drop=True, inplace=True)
evaluation

,Algorithms,Accuracy
0,Logistic Regression,0.799546
1,Support Vector Machines,0.797842
2,Random Forest,0.785349
3,KNN,0.764338
4,Decision Tree,0.724020


### Conclusion 
+ Logistic Regression and Support Vector Machines typically yield the best accuracy for this dataset.
+ Logistic Regression is preferred as it is highly interpretable, allowing us to understand exactly which features drive customer churn.

### Advanced Optimization: Handling Class Imbalance with SMOTE
* **Observation:** The original training data is imbalanced (~73% No, 27% Yes). We will apply SMOTE to generate synthetic minority samples.
* We strictly apply SMOTE **only to the training data** to prevent data leakage into the test set.

In [68]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Fit and apply SMOTE only on the transformed training data
X_train_balanced, Y_train_balanced = smote.fit_resample(X_train_transformed, Y_train)

print(f"Original Training Target Counts:\n{Y_train.value_counts()}\n")
print(f"Balanced Training Target Counts (After SMOTE):\n{Y_train_balanced.value_counts()}")

Original Training Target Counts:
Churn
0    3880
1    1402
Name: count, dtype: int64

Balanced Training Target Counts (After SMOTE):
Churn
0    3880
1    3880
Name: count, dtype: int64


### Hyperparameter Tuning: Gradient Boosting Classifier
* We will use `RandomizedSearchCV` to find the optimal parameters for our Gradient Boosting model.
* By restricting `max_depth` to smaller numbers (3, 4, 5), we strictly **prevent overfitting**.

In [69]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the Base Model
GBDTclf = GradientBoostingClassifier(random_state=42)

# 2. Define the Hyperparameter Grid to search through
param_grid = {
    'n_estimators': [100, 200, 300],        # Number of boosting stages
    'learning_rate': [0.01, 0.05, 0.1, 0.2],# Step size shrinkage
    'max_depth': [3, 4, 5],                 # Restricting depth prevents OVERFITTING
    'min_samples_split': [2, 5, 10]         # Minimum samples required to split a node
}

# 3. Initialize RandomizedSearchCV (Tests 20 random combinations, using 3-fold cross-validation)
random_search = RandomizedSearchCV(estimator=GBDTclf, param_distributions=param_grid, 
                                   n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

# 4. Train the model using the BALANCED SMOTE data
print("Searching for the best parameters... (This may take a minute)")
random_search.fit(X_train_balanced, Y_train_balanced)

print("Best Parameters Found:")
print(random_search.best_params_)

Searching for the best parameters... (This may take a minute)
Best Parameters Found:
{'n_estimators': 200, 'min_samples_split': 10, 'max_depth': 5, 'learning_rate': 0.1}


In [70]:
### Evaluate the Tuned Model on the UNSEEN Test Data
best_GBDT = random_search.best_estimator_

# Make predictions on the unseen test set
y_pred_tuned = best_GBDT.predict(X_test_transformed)

# Calculate Accuracy
from sklearn import metrics
acc_tuned = metrics.accuracy_score(Y_test, y_pred_tuned)
print('Optimized GBDT Accuracy Score: ', acc_tuned)

# Classification Report
print("\nClassification Report:")
print(metrics.classification_report(Y_test, y_pred_tuned))

Optimized GBDT Accuracy Score:  0.7649063032367973

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.80      0.83      1294
           1       0.55      0.68      0.60       467

    accuracy                           0.76      1761
   macro avg       0.71      0.74      0.72      1761
weighted avg       0.79      0.76      0.77      1761



In [71]:
### Update the Evaluation DataFrame with the Optimized Model
new_row = pd.DataFrame({
    'Algorithms': ['Optimized GBDT (SMOTE)'], 
    'Accuracy': [acc_tuned]
})

# Append and sort
evaluation = pd.concat([evaluation, new_row], ignore_index=True)
evaluation.sort_values(by='Accuracy', ascending=False, inplace=True)
evaluation.reset_index(drop=True, inplace=True)

display(evaluation)

,Algorithms,Accuracy
0,Logistic Regression,0.799546
1,Support Vector Machines,0.797842
2,Random Forest,0.785349
3,Optimized GBDT (SMOTE),0.764906
4,KNN,0.764338
5,Decision Tree,0.724020


### Advanced Evaluation and Comparison
* Relying solely on Accuracy is dangerous for imbalanced datasets.
* We will evaluate all models across Accuracy, Precision, Recall, and the F1-Score.
* The models will be ranked by **F1-Score**, which is the harmonic mean of Precision and Recall, providing the truest measure of a model's real-world business value.

In [72]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dictionary of all our trained models
trained_models = {
    'Logistic Regression': LogReg,
    'KNN': KNNclf,
    'Decision Tree': DTclf,
    'Random Forest': RFclf,
    'Support Vector Machines': SVMclf,
    'Optimized GBDT (SMOTE)': best_GBDT
}

evaluation_results = []

# Loop through each model and calculate all 4 metrics
for name, model in trained_models.items():
    # Make predictions on the test set
    y_pred = model.predict(X_test_transformed)
    
    # Calculate metrics
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred)
    rec = recall_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    
    # Append to list
    evaluation_results.append({
        'Algorithm': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

# Convert to DataFrame
comprehensive_eval = pd.DataFrame(evaluation_results)

# Sort by F1-Score (The ultimate balanced metric)
comprehensive_eval.sort_values(by='F1-Score', ascending=False, inplace=True)
comprehensive_eval.reset_index(drop=True, inplace=True)

display(comprehensive_eval)

,Algorithm,Accuracy,Precision,Recall,F1-Score
0,Optimized GBDT (SMOTE),0.764906,0.545769,0.676660,0.604207
1,Logistic Regression,0.799546,0.643939,0.546039,0.590962
2,Support Vector Machines,0.797842,0.663717,0.481799,0.558313
3,KNN,0.764338,0.556769,0.546039,0.551351
4,Random Forest,0.785349,0.622590,0.483940,0.544578
5,Decision Tree,0.724020,0.480331,0.496788,0.488421


### Exporting Model

In [73]:
import os
import pickle

# Define your specific folder path (using 'r' so the backslashes are read correctly)
save_dir = r"D:\Customer-Churn\Models"

# Create the folder if it doesn't already exist (prevents FileNotFoundError)
os.makedirs(save_dir, exist_ok=True)

# 1. Save the Optimized GBDT (The F1 & Recall Champion)
with open(os.path.join(save_dir, 'churn_model_gbdt.pkl'), 'wb') as f:
    pickle.dump(best_GBDT, f)

# 1b. Save Logistic Regression as a backup (The Pure Accuracy Champion)
with open(os.path.join(save_dir, 'churn_model_lr.pkl'), 'wb') as f:
    pickle.dump(LogReg, f)

# 2. Save the perfectly fitted Scaler
with open(os.path.join(save_dir, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scale, f)

# 3. Save the exact column layout 
with open(os.path.join(save_dir, 'model_columns.pkl'), 'wb') as f:
    pickle.dump(colos, f)

print(f"✅ Sprint 3 Complete! Models, Scaler, and Columns successfully exported to:\n{save_dir}")

✅ Sprint 3 Complete! Models, Scaler, and Columns successfully exported to:
D:\Customer-Churn\Models
